# Calibration

Fit Vasicek and CIR to a **Bloomberg EUR OIS** yield curve
(`data/bloomberg_eur_ois_curve.csv`).


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np

from calibration.calibrator import calibrate_cir, calibrate_vasicek
from calibration.yield_curve import YieldCurve
from rates import vasicek

FIGURES = ROOT / "figures"
FIGURES.mkdir(exist_ok=True)

BBG = ROOT / "data" / "bloomberg_eur_ois_curve.csv"
SAMPLE = ROOT / "data" / "sample_yield_curve.csv"
curve_path = BBG if BBG.exists() else SAMPLE
print("Using curve:", curve_path.relative_to(ROOT))


In [ ]:
curve = YieldCurve.from_csv(curve_path)
print("Maturities:", curve.maturities)
print("Zero rates:", curve.zero_rates)

vasicek_fit = calibrate_vasicek(curve)
cir_fit = calibrate_cir(curve)

print("\nVasicek params:", vasicek_fit.params)
print(f"Vasicek RMSE: {vasicek_fit.rmse:.6f}")
print("\nCIR params:", cir_fit.params)
print(f"CIR RMSE: {cir_fit.rmse:.6f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, fit in zip(axes, [vasicek_fit, cir_fit]):
    ax.plot(fit.maturities, fit.market_prices, "o", label="Market")
    ax.plot(fit.maturities, fit.fitted_prices, "-", label="Model")
    ax.set_title(fit.model_name)
    ax.set_xlabel("Maturity")
    ax.set_ylabel("Bond price P(0,T)")
    ax.legend()
    ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES / "calibration_fit.png", dpi=150)
plt.show()


In [ ]:
vp = vasicek_fit.params
a, b, sigma, r0 = vp["a"], vp["b"], vp["sigma"], vp["r0"]
T = curve.maturities

base_prices = np.array([vasicek.bond_price(a, b, sigma, r0, 0.0, t) for t in T])
up_prices = np.array([vasicek.bond_price(a, b, sigma * 1.1, r0, 0.0, t) for t in T])
down_prices = np.array([vasicek.bond_price(a, b, sigma * 0.9, r0, 0.0, t) for t in T])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(T, base_prices, label="Calibrated sigma")
ax.plot(T, up_prices, label="sigma +10%")
ax.plot(T, down_prices, label="sigma -10%")
ax.set_xlabel("Maturity")
ax.set_ylabel("Bond price")
ax.set_title("Vasicek sensitivity to volatility")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES / "calibration_sensitivity.png", dpi=150)
plt.show()
